# Laboratorio 3 — Análisis de Ventas (Customer / Invoice)

Dataset: `Customer.csv` (59 clientes) e `Invoice.csv` (412 facturas).
Cada punto del enunciado está resuelto en su propia sección.

## 1. Carga de los archivos

Ambos CSV están en la misma carpeta que este notebook, así que se leen con
`pd.read_csv()` usando la ruta relativa.

In [1]:
import pandas as pd

customer = pd.read_csv("Customer.csv")
invoice  = pd.read_csv("Invoice.csv")

print("Customer:", customer.shape)
print("Invoice :", invoice.shape)
customer.head()

Customer: (59, 13)
Invoice : (412, 9)


,CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId
0,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br,3
1,2,Leonie,Köhler,NaN,Theodor-Heuss-Straße 34,Stuttgart,NaN,Germany,70174,+49 0711 2842222,NaN,leonekohler@surfeu.de,5
2,3,François,Tremblay,NaN,1498 rue Bélanger,Montréal,QC,Canada,H2G 1A7,+1 (514) 721-4711,NaN,ftremblay@gmail.com,3
3,4,Bjørn,Hansen,NaN,Ullevålsveien 14,Oslo,NaN,Norway,0171,+47 22 44 22 22,NaN,bjorn.hansen@yahoo.no,4
4,5,František,Wichterlová,JetBrains s.r.o.,Klanova 9/506,Prague,NaN,Czech Republic,14700,+420 2 4172 5555,+420 2 4172 5555,frantisekw@jetbrains.com,4


In [2]:
invoice.head()

,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,2,2009-01-01 00:00:00,Theodor-Heuss-Straße 34,Stuttgart,NaN,Germany,70174,1.98
1,2,4,2009-01-02 00:00:00,Ullevålsveien 14,Oslo,NaN,Norway,0171,3.96
2,3,8,2009-01-03 00:00:00,Grétrystraat 63,Brussels,NaN,Belgium,1000,5.94
3,4,14,2009-01-06 00:00:00,8210 111 ST NW,Edmonton,AB,Canada,T6G 2C7,8.91
4,5,23,2009-01-11 00:00:00,69 Salem Street,Boston,MA,USA,2113,13.86


## 2. Nulos y duplicados

Revisamos cuántos valores faltantes hay por columna y si existen filas repetidas.

In [3]:
print("--- Nulos en Customer ---")
print(customer.isnull().sum())

print("\n--- Nulos en Invoice ---")
print(invoice.isnull().sum())

--- Nulos en Customer ---
CustomerId       0
FirstName        0
LastName         0
Company         49
Address          0
City             0
State           29
Country          0
PostalCode       4
Phone            1
Fax             47
Email            0
SupportRepId     0
dtype: int64

--- Nulos en Invoice ---
InvoiceId              0
CustomerId             0
InvoiceDate            0
BillingAddress         0
BillingCity            0
BillingState         202
BillingCountry         0
BillingPostalCode     28
Total                  0
dtype: int64


In [4]:
print("Duplicados en Customer:", customer.duplicated().sum())
print("Duplicados en Invoice :", invoice.duplicated().sum())

# Duplicados por la llave primaria (debe ser 0)
print("CustomerId repetidos:", customer["CustomerId"].duplicated().sum())
print("InvoiceId repetidos :", invoice["InvoiceId"].duplicated().sum())

Duplicados en Customer: 0
Duplicados en Invoice : 0
CustomerId repetidos: 0
InvoiceId repetidos : 0


**Resultado:** no hay filas duplicadas en ninguna tabla.
Los nulos están solo en columnas descriptivas opcionales (`Company`, `State`,
`Fax`, `PostalCode`, `Phone`, `BillingState`, `BillingPostalCode`).
Ninguna columna que usaremos para los cálculos (`Total`, `CustomerId`,
`BillingCountry`, `InvoiceDate`) tiene nulos, así que **no se elimina ni se
imputa nada**.

## 3. Tipos de dato — convertir `InvoiceDate`

`read_csv` no interpreta fechas automáticamente: `InvoiceDate` llega como
`object` (texto). La convertimos con `pd.to_datetime()` para poder extraer año
y mes más adelante.

In [5]:
print("Antes:", invoice["InvoiceDate"].dtype)

invoice["InvoiceDate"] = pd.to_datetime(invoice["InvoiceDate"])

print("Despues:", invoice["InvoiceDate"].dtype)
print("Rango de fechas:", invoice["InvoiceDate"].min(), "->", invoice["InvoiceDate"].max())

Antes: object
Despues: datetime64[ns]
Rango de fechas: 2009-01-01 00:00:00 -> 2013-12-22 00:00:00


## 4. Top 10 países por ingresos

Agrupamos por `BillingCountry`, sumamos `Total` y ordenamos de mayor a menor.
Luego calculamos qué porcentaje del ingreso global aporta el primer país.

In [6]:
ingresos_pais = invoice.groupby("BillingCountry")["Total"].sum().sort_values(ascending=False)

top10 = ingresos_pais.head(10)
print(top10)

BillingCountry
USA               523.06
Canada            303.96
France            195.10
Brazil            190.10
Germany           156.48
United Kingdom    112.86
Czech Republic     90.24
Portugal           77.24
India              75.26
Chile              46.62
Name: Total, dtype: float64


In [7]:
total_global = ingresos_pais.sum()
lider = top10.index[0]
pct = top10.iloc[0] / total_global * 100

print(f"Ingreso global: {total_global:,.2f}")
print(f"Pais lider: {lider} -> {top10.iloc[0]:,.2f} ({pct:.2f}% del total)")

Ingreso global: 2,328.60
Pais lider: USA -> 523.06 (22.46% del total)


**Lectura:** EE.UU. encabeza la lista, pero con un porcentaje relativamente bajo
del total: el ingreso está **repartido entre muchos países**, no concentrado en uno.

## 5. Ticket promedio global y comparación EE.UU. vs Francia

El *ticket promedio* es el monto promedio por factura: `Total.mean()`.

In [8]:
ticket_global = invoice["Total"].mean()
print(f"Ticket promedio global: {ticket_global:.2f}")

Ticket promedio global: 5.65


In [9]:
ticket_usa    = invoice.loc[invoice["BillingCountry"] == "USA",    "Total"].mean()
ticket_francia = invoice.loc[invoice["BillingCountry"] == "France", "Total"].mean()

print(f"USA    : {ticket_usa:.2f}")
print(f"France : {ticket_francia:.2f}")
print(f"Diferencia (USA - France): {ticket_usa - ticket_francia:.2f}")

USA    : 5.75
France : 5.57
Diferencia (USA - France): 0.17


**Lectura:** los dos tickets promedio son muy parecidos entre sí y respecto al
global. EE.UU. no factura más *por compra*, sino que tiene **más clientes y más
facturas**; ahí está su ventaja en ingresos.

## 6. Tabla por país con `.agg()` y columnas nombradas

Una sola instrucción que devuelve ingresos totales, número de facturas y ticket
promedio usando *named aggregation* (`nombre=("columna", "funcion")`).

In [10]:
resumen_pais = (
    invoice.groupby("BillingCountry")
           .agg(ingresos_totales=("Total", "sum"),
                num_facturas=("InvoiceId", "count"),
                ticket_promedio=("Total", "mean"))
           .sort_values("ingresos_totales", ascending=False)
           .round(2)
)

resumen_pais.head(10)

,ingresos_totales,num_facturas,ticket_promedio
BillingCountry,,,
USA,523.06,91,5.75
Canada,303.96,56,5.43
France,195.10,35,5.57
Brazil,190.10,35,5.43
Germany,156.48,28,5.59
United Kingdom,112.86,21,5.37
Czech Republic,90.24,14,6.45
Portugal,77.24,14,5.52
India,75.26,13,5.79


## 7. Merge y comparación `Country` vs `BillingCountry`

Unimos las dos tablas por `CustomerId` y comparamos el país registrado del
cliente contra el país de facturación.

In [11]:
df = invoice.merge(customer, on="CustomerId", how="inner")
print("Filas tras el merge:", df.shape)

discrepancias = df[df["BillingCountry"] != df["Country"]]
print("Facturas con pais distinto al del cliente:", len(discrepancias))

discrepancias[["InvoiceId", "CustomerId", "Country", "BillingCountry"]]

Filas tras el merge: (412, 21)
Facturas con pais distinto al del cliente: 0


,InvoiceId,CustomerId,Country,BillingCountry


**Resultado:** 0 discrepancias. En las 412 facturas el país de facturación
coincide siempre con el país del cliente, es decir, **la dirección de facturación
se copia del perfil del cliente**. Esto valida la consistencia de las dos tablas.

## 8. Clientes sin facturas

Aquí el tipo de unión correcto es **LEFT JOIN** desde `Customer` hacia `Invoice`:
así conservamos a todos los clientes, incluso a los que no aparecen en `Invoice`.
Los que nunca facturaron quedan con `InvoiceId` nulo.

In [12]:
left = customer.merge(invoice, on="CustomerId", how="left", indicator=True)

sin_factura = left[left["_merge"] == "left_only"]
print("Clientes sin ninguna factura:", sin_factura["CustomerId"].nunique())

sin_factura[["CustomerId", "FirstName", "LastName", "Country"]]

Clientes sin ninguna factura: 0


,CustomerId,FirstName,LastName,Country


**Resultado:** ningún cliente quedó sin facturas — los 59 clientes de `Customer`
tienen al menos una compra. (`indicator=True` crea la columna `_merge`, que marca
`left_only` cuando no hubo coincidencia.)

## 9. Top 10 clientes por gasto total (con nombre completo)

Construimos el nombre completo concatenando `FirstName` y `LastName`, y sumamos
el `Total` de sus facturas.

In [13]:
df["NombreCompleto"] = df["FirstName"] + " " + df["LastName"]

top_clientes = (
    df.groupby(["CustomerId", "NombreCompleto", "Country"])
      .agg(gasto_total=("Total", "sum"),
           num_facturas=("InvoiceId", "count"))
      .sort_values("gasto_total", ascending=False)
      .head(10)
      .round(2)
      .reset_index()
)

top_clientes

,CustomerId,NombreCompleto,Country,gasto_total,num_facturas
0,6,Helena Holý,Czech Republic,49.62,7
1,26,Richard Cunningham,USA,47.62,7
2,57,Luis Rojas,Chile,46.62,7
3,45,Ladislav Kovács,Hungary,45.62,7
4,46,Hugh O'Reilly,Ireland,45.62,7
5,28,Julia Barnett,USA,43.62,7
6,37,Fynn Zimmermann,Germany,43.62,7
7,24,Frank Ralston,USA,43.62,7
8,7,Astrid Gruber,Austria,42.62,7
9,25,Victor Stevens,USA,42.62,7


## 10. Tabla pivote: año (filas) × país (columnas)

Extraemos el año de `InvoiceDate` (ya convertido en el punto 3) con el accesor
`.dt.year` y armamos el pivote con la suma de `Total`.

In [14]:
invoice["Anio"] = invoice["InvoiceDate"].dt.year

pivote = pd.pivot_table(invoice,
                        index="Anio",
                        columns="BillingCountry",
                        values="Total",
                        aggfunc="sum",
                        fill_value=0).round(2)

pivote

BillingCountry,Argentina,Australia,Austria,Belgium,Brazil,Canada,Chile,Czech Republic,Denmark,Finland,...,Ireland,Italy,Netherlands,Norway,Poland,Portugal,Spain,Sweden,USA,United Kingdom
Anio,,,,,,,,,,,,,,,,,,,,,
2009,0.00,11.88,1.98,6.93,37.62,57.42,15.84,10.89,5.94,8.91,...,6.93,1.98,8.91,10.89,15.84,11.88,0.99,5.94,103.95,25.74
2010,11.88,0.99,27.77,0.00,41.60,76.26,17.91,9.90,6.93,0.00,...,0.00,10.89,1.98,0.00,8.91,6.93,1.98,7.93,102.98,30.69
2011,0.99,1.98,0.00,24.75,19.80,55.44,5.94,12.87,0.00,15.88,...,32.75,0.00,12.90,17.84,0.00,8.91,22.77,0.00,103.01,17.82
2012,0.00,22.77,11.88,0.00,53.46,42.57,6.93,19.83,15.84,0.99,...,0.00,15.84,0.99,8.91,11.88,24.77,0.00,24.75,127.98,9.90
2013,24.75,0.00,0.99,5.94,37.62,72.27,0.00,36.75,8.91,15.84,...,5.94,8.91,15.84,1.98,0.99,24.75,11.88,0.00,85.14,28.71


In [15]:
# Version resumida: solo los 8 paises de mayor ingreso, mas facil de leer
top8 = ingresos_pais.head(8).index
pivote[top8]

BillingCountry,USA,Canada,France,Brazil,Germany,United Kingdom,Czech Republic,Portugal
Anio,,,,,,,,
2009,103.95,57.42,35.64,37.62,53.46,25.74,10.89,11.88
2010,102.98,76.26,39.60,41.60,25.74,30.69,9.90,6.93
2011,103.01,55.44,42.61,19.80,48.57,17.82,12.87,8.91
2012,127.98,42.57,36.66,53.46,18.81,9.90,19.83,24.77
2013,85.14,72.27,40.59,37.62,9.90,28.71,36.75,24.75


## 11. Tasa de recompra

Porcentaje de clientes con más de una factura.

In [16]:
facturas_por_cliente = invoice.groupby("CustomerId")["InvoiceId"].count()

recompradores = (facturas_por_cliente > 1).sum()
total_clientes = customer["CustomerId"].nunique()
tasa = recompradores / total_clientes * 100

print(f"Clientes totales        : {total_clientes}")
print(f"Clientes con >1 factura : {recompradores}")
print(f"Tasa de recompra        : {tasa:.2f}%")
print(f"Facturas por cliente (promedio): {facturas_por_cliente.mean():.1f}")

Clientes totales        : 59
Clientes con >1 factura : 59
Tasa de recompra        : 100.00%
Facturas por cliente (promedio): 7.0


**Resultado:** 100%. Todos los clientes compraron más de una vez (en promedio
≈7 facturas cada uno). El negocio **no tiene problema de retención**; el reto
está en el monto por compra, no en que el cliente vuelva.

# 12. Reportes libres

Cuatro análisis adicionales sobre el dataset limpio. Cada uno indica **qué se
encontró**, **cómo se comprobó** y la **recomendación de negocio**.

### Reporte 1 — Concentración de ingresos (¿existe riesgo de dependencia?)

Pregunta: ¿el negocio depende de unos pocos clientes grandes?

In [17]:
gasto = df.groupby("NombreCompleto")["Total"].sum().sort_values(ascending=False)

share_top1  = gasto.iloc[0]  / gasto.sum() * 100
share_top10 = gasto.head(10).sum() / gasto.sum() * 100

print(f"Cliente #1 aporta      : {share_top1:.2f}% del ingreso")
print(f"Top 10 clientes aportan: {share_top10:.2f}% del ingreso")
print(f"Gasto max: {gasto.max():.2f} | min: {gasto.min():.2f} | promedio: {gasto.mean():.2f}")

Cliente #1 aporta      : 2.13% del ingreso
Top 10 clientes aportan: 19.38% del ingreso
Gasto max: 49.62 | min: 36.64 | promedio: 39.47


**Qué se encontró:** los 10 mejores clientes (17% de la base) generan apenas
~19% del ingreso, y el mejor cliente solo ~2%. Muy lejos del clásico 80/20.

**Cómo se comprobó:** suma de `Total` por cliente dividida entre el ingreso
global; se contrastaron máximo, mínimo y promedio para ver la dispersión, que
resultó baja.

**Recomendación:** no hay riesgo de dependencia de clientes clave, así que
**no conviene invertir en un programa VIP**. Como todos gastan parecido, el
crecimiento debe venir de subir el ticket promedio de *toda* la base
(bundles, precios por volumen) o de captar clientes nuevos.

### Reporte 2 — Evolución anual de ingresos

Pregunta: ¿el negocio está creciendo, estancado o cayendo?

In [18]:
anual = (invoice.groupby("Anio")
                .agg(ingresos=("Total", "sum"),
                     facturas=("InvoiceId", "count"),
                     ticket=("Total", "mean"))
                .round(2))

anual["var_%"] = (anual["ingresos"].pct_change() * 100).round(2)
anual

,ingresos,facturas,ticket,var_%
Anio,,,,
2009,449.46,83,5.42,NaN
2010,481.45,83,5.80,7.12
2011,469.58,83,5.66,-2.47
2012,477.53,83,5.75,1.69
2013,450.58,80,5.63,-5.64


**Qué se encontró:** los ingresos se mueven en una banda muy estrecha durante
los 5 años (≈449–481 por año), con variaciones interanuales de apenas unos
puntos porcentuales y sin tendencia clara. El negocio está **plano**.

**Cómo se comprobó:** agregación anual de ingresos, facturas y ticket, más
`pct_change()` para medir el crecimiento año contra año. Tanto el número de
facturas como el ticket se mantienen estables, así que el estancamiento no lo
explica ninguna de las dos variables por separado.

**Recomendación:** la operación actual está en equilibrio y no crecerá sola.
Cualquier meta de crecimiento exige una **acción deliberada** (nuevos mercados,
catálogo o promociones), y con esta serie histórica tan estable, cualquier
cambio superior a ~±8% después de una campaña puede atribuirse con confianza
a esa campaña.

### Reporte 3 — Estacionalidad mensual

Pregunta: ¿hay meses fuertes y débiles que sirvan para planificar campañas?

In [19]:
invoice["Mes"] = invoice["InvoiceDate"].dt.month

mensual = (invoice.groupby("Mes")
                  .agg(ingresos=("Total", "sum"),
                       facturas=("InvoiceId", "count"))
                  .round(2))

mensual["%_del_total"] = (mensual["ingresos"] / mensual["ingresos"].sum() * 100).round(2)
print(mensual)

print(f"\nMes mas alto: {mensual['ingresos'].idxmax()} | Mes mas bajo: {mensual['ingresos'].idxmin()}")
print(f"Brecha alto vs bajo: {mensual['ingresos'].max() / mensual['ingresos'].min():.2f}x")

     ingresos  facturas  %_del_total
Mes                                 
1      201.12        34         8.64
2      187.20        33         8.04
3      195.10        35         8.38
4      198.14        33         8.51
5      193.10        35         8.29
6      201.10        35         8.64
7      190.10        35         8.16
8      198.10        35         8.51
9      196.20        33         8.43
10     193.10        35         8.29
11     186.24        34         8.00
12     189.10        35         8.12

Mes mas alto: 1 | Mes mas bajo: 11
Brecha alto vs bajo: 1.08x


**Qué se encontró:** los 12 meses aportan entre 8.0% y 8.6% del ingreso total y
el mejor mes solo supera al peor por 1.08x. **No hay estacionalidad relevante.**

**Cómo se comprobó:** se agruparon los 5 años por mes calendario y se calculó
la participación de cada mes; ningún mes se despega del 8.3% que correspondería
a un reparto perfectamente uniforme.

**Recomendación:** no gastar presupuesto en campañas de temporada — no hay un
pico natural que aprovechar ni un valle que rescatar. Conviene un
**presupuesto de marketing constante**, y usar los meses indistintamente para
hacer pruebas A/B de precios, ya que la estacionalidad no contamina la
comparación.

### Reporte 4 — Desempeño de los representantes de ventas

Pregunta: ¿los agentes de soporte (`SupportRepId`) rinden parejo?

In [20]:
reps = (df.groupby("SupportRepId")
          .agg(ingresos=("Total", "sum"),
               clientes=("CustomerId", "nunique"),
               facturas=("InvoiceId", "count"))
          .round(2))

reps["ingreso_x_cliente"] = (reps["ingresos"] / reps["clientes"]).round(2)
reps.sort_values("ingresos", ascending=False)

,ingresos,clientes,facturas,ingreso_x_cliente
SupportRepId,,,,
3,833.04,21,146,39.67
4,775.40,20,140,38.77
5,720.16,18,126,40.01


**Qué se encontró:** hay diferencias en el ingreso total por representante,
pero se explican casi por completo por **cuántos clientes tiene asignados cada
uno**: al normalizar (`ingreso_x_cliente`) los tres quedan prácticamente iguales.

**Cómo se comprobó:** se compararon ingresos absolutos contra ingresos por
cliente; el ranking absoluto y el número de clientes asignados siguen el mismo
orden, mientras que el ingreso por cliente se aplana.

**Recomendación:** **no premiar ni penalizar por ingreso absoluto**, porque
mide el tamaño de la cartera y no el desempeño. La métrica justa de comisión es
el ingreso por cliente asignado; y si se quiere subir la facturación de un
representante, la palanca real es **reasignar cartera**, no exigir más ventas.

---
## Conclusión general

El dataset está **limpio y consistente**: sin duplicados, sin clientes huérfanos
y con el país de facturación siempre alineado al del cliente. El negocio es
**estable pero plano**: ingresos sin crecimiento en 5 años, sin estacionalidad,
sin concentración de clientes y con 100% de recompra. La palanca de crecimiento
no es la retención (ya está al máximo) sino **subir el ticket promedio** y
**ampliar la base de clientes**.